# 잘못 귀속 안전 조건의 판단 재료

D-67 결정 3의 안전 조건("발언자 불일치" 진술 비율의 증가폭 상한)을 정하기 전에 지금 기록으로 값의 크기와 구간을 본다. 새 API 호출은 없다. 판정을 바꾸지 않으며 EXP-019·EXP-020의 판정은 그대로다. 질의·답변 원문은 출력하지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from rag.eval.bootstrap import paired_bootstrap
from rag.eval.metrics import load_gold
from rag.gen_eval.common import read_jsonl
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
P = "data/gen_eval/judge/hybrid-cond-filter-law-ms256-k20-ce-"
RUNS = {
    "EXP-018 j_v11": ROOT / (P + "v3-dev-full-deepseek_go_mixed-j_v11.jsonl"),
    "EXP-019 j_v11": ROOT / (P + "v4-dev-full-spk-head-deepseek_go_mixed-j_v11.jsonl"),
    "EXP-019 j_v12": ROOT / (P + "v4-dev-full-spk-head-deepseek_go_mixed-j_v12.jsonl"),
    "EXP-020 j_v12": ROOT / (P + "v6-dev-full-spk-head-deepseek_go_mixed-j_v12.jsonl"),
}
PAIRS = [("EXP-018 j_v11", "EXP-019 j_v11"), ("EXP-019 j_v12", "EXP-020 j_v12")]
MISMATCH = "발언자 불일치"

In [3]:
def query_rates(path: Path) -> pd.DataFrame:
    """질의마다 진술 수와 이유 첫머리가 발언자 불일치인 진술 수. 진술이 없는 질의는 NaN이다."""
    rows = {}
    for r in read_jsonl(path):
        vs = (r.get("faithfulness") or {}).get("verdicts") or []
        hit = sum(v["reason"].startswith(MISMATCH) for v in vs)
        any_hit = sum(MISMATCH in v["reason"] for v in vs)
        rows[r["qid"]] = {"n": len(vs), "hit": hit, "any": any_hit, "rate": hit / len(vs) if vs else np.nan}
    return pd.DataFrame(rows).T

## 값의 크기

지표 후보는 질의마다 불일치 진술 수 / 진술 수를 낸 값의 평균이다(과잉 거부율 D-52와 같은 꼴). 진술이 없는 질의는 함께 뺀다.

In [4]:
rates = {k: query_rates(p) for k, p in RUNS.items()}
rows = []
for k, d in rates.items():
    done = d.dropna(subset=["rate"])
    rows.append({"실행": k, "질의": len(done), "진술": int(done.n.sum()), "불일치(첫머리)": int(done.hit.sum()), "불일치(이유 어디든)": int(done["any"].sum()), "불일치 있는 질의": int((done.hit > 0).sum()), "평균 비율": round(done.rate.mean(), 4), "합친 비율": round(done.hit.sum() / done.n.sum(), 4)})
pd.DataFrame(rows)

,실행,질의,진술,불일치(첫머리),불일치(이유 어디든),불일치 있는 질의,평균 비율,합친 비율
0,EXP-018 j_v11,300,753,2,2,2,0.0033,0.0027
1,EXP-019 j_v11,297,746,19,19,14,0.0261,0.0255
2,EXP-019 j_v12,297,746,19,19,14,0.0261,0.0255
3,EXP-020 j_v12,307,868,56,56,46,0.0607,0.0645


## 차이와 구간

회의 단위 paired bootstrap(판정 절과 같은 10,000회, seed)이다. 두 실행 모두 값이 있는 질의만 쓴다.

In [5]:
cfg = load_cfg(["+exp=current"])
bs = cfg.eval.bootstrap
groups = load_gold(cfg, cfg.gen_eval.queries)[bs.unit]
rows = []
for a, b in PAIRS:
    both = rates[a].rate.notna() & rates[b].rate.notna()
    d = (rates[b].rate - rates[a].rate)[both]
    pt, lo, hi = paired_bootstrap(d.to_numpy(float), groups.loc[d.index].to_numpy(), bs.n_resamples, bs.seed, 0.05)
    rows.append({"비교": f"{a} → {b}", "질의": int(both.sum()), "차이": round(pt, 4), "하한": round(lo, 4), "상한": round(hi, 4)})
diffs = pd.DataFrame(rows)
diffs

,비교,질의,차이,하한,상한
0,EXP-018 j_v11 → EXP-019 j_v11,288,0.0234,0.0080,0.0404
1,EXP-019 j_v12 → EXP-020 j_v12,291,0.0339,0.0153,0.0527


## 허용폭 후보에 대면

값마다 두 비교가 상한 조건을 통과하는지 본다. 같은 설정 재실행의 반폭은 아직 재지 않았다(발언자 머리 줄 뒤로 같은 설정을 다시 돌린 기록이 없다).

In [6]:
cands = [0.0, 0.005, 0.01, 0.02, 0.03, 0.05]
passed = {f"{t}": (diffs["상한"] <= t).tolist() for t in cands}
pd.DataFrame(passed, index=diffs["비교"])

,0.0,0.005,0.01,0.02,0.03,0.05
비교,,,,,,
EXP-018 j_v11 → EXP-019 j_v11,False,False,False,False,False,True
EXP-019 j_v12 → EXP-020 j_v12,False,False,False,False,False,False


## 측정값의 한계

EXP-020 판독(서브에이전트 1회, 표본): 불일치 판정 30개 중 23개가 맞고 7개는 오판이다. 이름 답변 표본의 잘못 귀속 7개 중 judge가 불일치로 잡은 것은 3개, 다른 이유로 0을 준 것이 1개, 놓친 것이 3개다. 그래서 이 지표는 잘못 귀속 자체가 아니라 judge가 불일치로 판정한 비율이고, 같은 judge로 기준과 후보를 재서 증가 여부를 본다.